# Exploratory Data Analysis with Pyspark and Spark SQL

The following notebook utilizes New York City taxi data from [TLC Trip Record Data](https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page)

## Instructions

- Load and explore nyc taxi data from january 0f 2019. The exercises can be executed using pyspark or spark sql (a subset of the questions will be re-answered using the language not chosen for the  main work).
- Load the zone lookup table to answer the questions about the nyc boroughs.  
- Load nyc taxi data from January of 2025 and compare data.  
- With any remaining time, work on the where to go from here section.
- Note: the initial lab is opened as read only. To save work completed utilize the `save notebook as` option and give the lab a new name.

In [1]:
import requests

# start a spark session and create a spark context
from pyspark.sql import SparkSession
spark = SparkSession.builder \
    .appName("nyc_taxi") \
    .getOrCreate()

sc = spark.sparkContext

In [2]:
# set dl url for January 2019 trip data
download_url = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2019-01.parquet'

# get the data
response = requests.get(download_url)

# check that response was good and save the data
jan_2019_trip_data = "yellow_tripdata_2019-01.parquet"
if response.status_code == 200:
    # Changed 'response' to 'data' here
    with open(jan_2019_trip_data, "wb") as f:
        f.write(response.content)


In [3]:
# create the dataframe
df_trips = spark.read.parquet(jan_2019_trip_data)

# A brief note on handling data sources in spark

The command above works well for loading data from parquet files because parquet is a self descibing file format, meaning that the metadata needed to build the dataframe is included directly in the format. However, when working with other formats such as csv or json, a schema must be provided or infered. In production code the schema should always be explicitly provided but during the data exploration phase it is acceptable to infer the schema, and when infering the schema it often best to use `.option("samplingRatio", <small-portion-of-data>)` to avoid using the entire dataset for schema inference.

```python
df_trips = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("samplingRatio", 0.01) \
    .load("large_dataset.csv")
```

In [5]:
# Show the dataframe
df_trips.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+
|       1| 2019-01-01 00:46:40|  2019-01-01 00:53:20|            1.0|          1.5|       1.0|                 N|         151|         239|           1|        7.0|  0.5|    0.5|      1.6

In [5]:
df_trips.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee',
 'ID']

## Lab

### Part 1
This section can be completed either using pyspark commands or sql commands ( There will be a section after in which a self-chosen subset of the questions are re-answered using the language not used for the main section. i.e. if pyspark is chosen for the main lab, sql should be used to repeat some of the questions. )

- Add a column that creates a unique key to identify each record in order to answer questions about individual trips
- Which trip has the highest passanger count
- What is the Average passanger count
- Shortest/longest trip by distance? by time?.
- busiest day/slowest single day
- busiest/slowest time of day ( you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
- On average which day of the week is slowest/busiest
- Does trip distance or num passangers affect tip amount
- What was the highest "extra" charge and which trip
- Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?

In [6]:
# Add a column that creates a unique key to identify each record in order to answer questions about individual trips
from pyspark.sql.functions import monotonically_increasing_id

df_trips = df_trips.withColumn('ID', monotonically_increasing_id())
df_trips.columns

['VendorID',
 'tpep_pickup_datetime',
 'tpep_dropoff_datetime',
 'passenger_count',
 'trip_distance',
 'RatecodeID',
 'store_and_fwd_flag',
 'PULocationID',
 'DOLocationID',
 'payment_type',
 'fare_amount',
 'extra',
 'mta_tax',
 'tip_amount',
 'tolls_amount',
 'improvement_surcharge',
 'total_amount',
 'congestion_surcharge',
 'airport_fee',
 'ID']

In [7]:
# Which trip has the highest passanger count
from pyspark.sql import functions as F
High_trip = df_trips.orderBy(F.desc('passenger_count')).limit(1)
High_trip.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|         ID|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       2| 2019-01-05 13:12:29|  2019-01-05 13:12:32|            9.0|          0.0|       5.0|                 N|          68|          68|           1

In [8]:
# What is the Average passanger count
Avg = df_trips.agg(F.avg('passenger_count')).collect()[0][0]
print(Avg)

1.5670317144945614


In [9]:
# Shortest/longest trip by distance? by time?.
Shortest_trip_by_distance = df_trips.orderBy('trip_distance').limit(1)
Shortest_trip_by_distance.show()

from pyspark.sql.functions import unix_timestamp, col

df_trips = df_trips.withColumn(
    'trip_duration_seconds',
    unix_timestamp(col('tpep_dropoff_datetime')) - unix_timestamp(col('tpep_pickup_datetime'))
)

Shortest_trip_by_time = df_trips.orderBy('trip_duration_seconds').limit(1)
Shortest_trip_by_time.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|         ID|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+
|       2| 2018-12-21 13:48:30|  2018-12-21 13:52:40|            3.0|          0.0|       1.0|                 N|         236|         236|           1

In [10]:
# busiest day/slowest single day
df_trips = df_trips.withColumn('pickup_date', F.to_date('tpep_pickup_datetime'))

busiest_day = df_trips.groupBy('pickup_date').count().orderBy(F.desc('count')).limit(1)
busiest_day.show()

slowest_day = df_trips.groupBy('pickup_date').count().orderBy('count').limit(1)
slowest_day.show()


+-----------+------+
|pickup_date| count|
+-----------+------+
| 2019-01-25|292499|
+-----------+------+

+-----------+-----+
|pickup_date|count|
+-----------+-----+
| 2019-02-23|    1|
+-----------+-----+



In [11]:
# busiest/slowest time of day (you may want to bucket these by hour or create timess such as morning, afternoon, evening, late night )
df_trips = df_trips.withColumn('pickup_hour', F.hour('tpep_pickup_datetime'))

df_trips = df_trips.withColumn(
    'time_of_day',
    F.when((F.col('pickup_hour') >= 5) & (F.col('pickup_hour') < 12), 'morning')
     .when((F.col('pickup_hour') >= 12) & (F.col('pickup_hour') < 17), 'afternoon')
     .when((F.col('pickup_hour') >= 17) & (F.col('pickup_hour') < 21), 'evening')
     .otherwise('late night')
)

busiest_period = df_trips.groupBy('time_of_day').count() \
                          .orderBy(F.desc('count')).limit(1)\
                          .withColumn('label', F.lit('Busiest'))
busiest_period.show()

slowest_period = df_trips.groupBy('time_of_day').count() \
                          .orderBy('count').limit(1) \
                          .withColumn('label', F.lit('Slowest'))
slowest_period.show()

#pour afficher en une fois
#result = busiest_period.union(slowest_period)
#result.show()

+-----------+-------+-------+
|time_of_day|  count|  label|
+-----------+-------+-------+
|  afternoon|2111999|Busiest|
+-----------+-------+-------+

+-----------+-------+-------+
|time_of_day|  count|  label|
+-----------+-------+-------+
| late night|1666910|Slowest|
+-----------+-------+-------+



In [ ]:
# On average which day of the week is slowest/busiest
busiest_day = df_trips.groupBy('day_of_week').count() \
                          .orderBy(F.desc('count')).limit(1)\
                          .withColumn('label', F.lit('Busiest'))
busiest_day.show()

slowest_day = df_trips.groupBy('day_of_week').count() \
                          .orderBy(F.desc('count')).limit(1)\
                          .withColumn('label', F.lit('Slowest'))
slowest_day.show()

In [12]:
# Does trip distance or num passangers affect tip amount
corr_distance_tip = df_trips.corr('trip_distance', 'tip_amount')
corr_passengers_tip = df_trips.corr('passenger_count', 'tip_amount')

print(f"Correlation trip_distance / tip_amount: {corr_distance_tip:.4f}")
print(f"Correlation passenger_count / tip_amount: {corr_passengers_tip:.4f}")

Correlation trip_distance / tip_amount: 0.5269
Correlation passenger_count / tip_amount: 0.0044


In [13]:
# What was the highest "extra" charge and which trip
Highest_extra = df_trips.orderBy(F.desc('extra')).limit(1)
Highest_extra.show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|         ID|trip_duration_seconds|pickup_date|pickup_hour|time_of_day|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------

In [ ]:
# Are there any datapoints that seem to be strange/outliers (make sure to explain your reasoning in a markdown cell)?
# check for trips with 0 distance and 0 duration but a very high fare
df_trips.filter(
    (F.col('trip_distance') == 0) &
    (F.col('tpep_pickup_datetime') == F.col('tpep_dropoff_datetime')) &
    (F.col('fare_amount') > 100)
).show()

outlier_count = df_trips.filter(
    (F.col('trip_distance') == 0) &
    (F.col('fare_amount') > 100)
).count()

print(f"Number of suspicious zero-distance, high-fare trips: {outlier_count}")

**Outlier explanation:**

Some rows show `trip_distance == 0` and identical pickup/dropoff timestamps (a 0-second trip), yet are billed with a `fare_amount` in the hundreds of thousands of dollars (e.g. $355,676.98) along with an unusually high `extra` charge. A real taxi ride typically costs between $5 and $100 and takes at least a few minutes, so a trip with no distance, no duration, and an extreme fare is not physically plausible. This combination strongly suggests a data entry error, meter malfunction, or corrupted record rather than an actual completed trip.

### Part 2

- Using the code for loading the first dataset as an example, load in the taxi zone lookup and answer the following questions
- which borough had most pickups? dropoffs?
- what are the busy/slow times by borough 
- what are the busiest days of the week by borough?
- what is the average trip distance by borough?
- what is the average trip fare by borough?
- highest/lowest faire amounts for a trip, what burough is associated with the each
- load the dataset from the most recently available january, is there a change to any of the average metrics.

In [14]:
# Using the code for loading the first dataset as an example, load in the taxi zone lookup
df_borough = spark.read.format("csv") \
    .option("header", "true") \
    .option("sep", ",") \
    .option("inferSchema", "true") \
    .load("taxi_zone_lookup.csv")

df_borough.show(5)

df_borough_pu = df_borough.withColumnRenamed('LocationID', 'PULocationID')
df_joined = df_trips.join(df_borough_pu, 'PULocationID')

df_joined.show(5)

+----------+-------------+--------------------+------------+
|LocationID|      Borough|                Zone|service_zone|
+----------+-------------+--------------------+------------+
|         1|          EWR|      Newark Airport|         EWR|
|         2|       Queens|         Jamaica Bay|   Boro Zone|
|         3|        Bronx|Allerton/Pelham G...|   Boro Zone|
|         4|    Manhattan|       Alphabet City| Yellow Zone|
|         5|Staten Island|       Arden Heights|   Boro Zone|
+----------+-------------+--------------------+------------+
only showing top 5 rows
+------------+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------+-----------+-----------+---------+--------------------+------------+
|PULocationID|VendorID|tpep_pickup_datetim

In [15]:
# which borough had most pickups? dropoffs?

# most pickups (using df_joined, joined on PULocationID)
print("Busiest borough by pickups:")
df_joined.groupBy('Borough').count().orderBy(F.desc('count')).limit(1).show()

df_borough_do = df_borough.withColumnRenamed('LocationID', 'DOLocationID')
df_joined_do = df_trips.join(df_borough_do, 'DOLocationID')

print("Busiest borough by dropoffs:")
df_joined_do.groupBy('Borough').count().orderBy(F.desc('count')).limit(1).show()

Busiest borough by pickups:
+---------+-------+
|  Borough|  count|
+---------+-------+
|Manhattan|6950965|
+---------+-------+

Busiest borough by dropoffs:
+---------+-------+
|  Borough|  count|
+---------+-------+
|Manhattan|6817355|
+---------+-------+



In [16]:
# what are the busy/slow times by borough
from pyspark.sql.window import Window

df_joined = df_joined.withColumn('pickup_hour', F.hour('tpep_pickup_datetime'))
hour_counts = df_joined.groupBy('Borough', 'pickup_hour').count()

window_busy = Window.partitionBy('Borough').orderBy(F.desc('count'))
print("Busiest hour per borough:")
hour_counts.withColumn('rank', F.row_number().over(window_busy)) \
           .filter(F.col('rank') == 1).drop('rank').show()

window_slow = Window.partitionBy('Borough').orderBy('count')
print("Slowest hour per borough:")
hour_counts.withColumn('rank', F.row_number().over(window_slow)) \
           .filter(F.col('rank') == 1).drop('rank').show()

Busiest hour per borough:
+-------------+-----------+------+
|      Borough|pickup_hour| count|
+-------------+-----------+------+
|        Bronx|          7|  1803|
|     Brooklyn|          8|  6935|
|          EWR|         15|    54|
|    Manhattan|         18|471539|
|          N/A|         19|   214|
|       Queens|         16| 29885|
|Staten Island|          8|    36|
|      Unknown|         18| 10751|
+-------------+-----------+------+

Slowest hour per borough:
+-------------+-----------+-----+
|      Borough|pickup_hour|count|
+-------------+-----------+-----+
|        Bronx|          3|  225|
|     Brooklyn|          3| 1919|
|          EWR|         23|    1|
|    Manhattan|          4|53447|
|          N/A|          6|   88|
|       Queens|          3| 3085|
|Staten Island|          1|    3|
|      Unknown|          4| 1465|
+-------------+-----------+-----+



In [17]:
# what are the busiest days of the week by borough?
df_joined = df_joined.withColumn('day_of_week', F.dayofweek('tpep_pickup_datetime'))
day_counts = df_joined.groupBy('Borough', 'day_of_week').count()

window_day = Window.partitionBy('Borough').orderBy(F.desc('count'))
print("Busiest day of week per borough:")
day_counts.withColumn('rank', F.row_number().over(window_day)) \
          .filter(F.col('rank') == 1).drop('rank').show()

Busiest day of week per borough:
+-------------+-----------+-------+
|      Borough|day_of_week|  count|
+-------------+-----------+-------+
|        Bronx|          5|   3121|
|     Brooklyn|          3|  15779|
|          EWR|          4|     83|
|    Manhattan|          5|1229554|
|          N/A|          3|    703|
|       Queens|          5|  78972|
|Staten Island|          6|     64|
|      Unknown|          5|  28929|
+-------------+-----------+-------+



In [18]:
# what is the average trip distance by borough?
df_joined.groupBy('Borough') \
         .agg(F.avg('trip_distance').alias('avg_distance')) \
         .orderBy(F.desc('avg_distance')) \
         .show()

+-------------+------------------+
|      Borough|      avg_distance|
+-------------+------------------+
|Staten Island|12.503601108033246|
|       Queens|11.283218499361993|
|        Bronx| 7.233194552098303|
|     Brooklyn| 4.787677275447492|
|          N/A| 3.193850899742941|
|          EWR| 2.641098654708519|
|      Unknown| 2.415464130400774|
|    Manhattan|2.2286693358402596|
+-------------+------------------+



In [19]:
# what is the average trip fare by borough?
df_joined.groupBy('Borough') \
         .agg(F.avg('fare_amount').alias('avg_fare')) \
         .orderBy(F.desc('avg_fare')) \
         .show()

+-------------+------------------+
|      Borough|          avg_fare|
+-------------+------------------+
|          EWR| 76.24024663677126|
|          N/A|  59.5731593830335|
|Staten Island|45.289861495844896|
|       Queens| 35.14462651722029|
|        Bronx| 26.26890543682963|
|     Brooklyn|18.649132800172286|
|      Unknown|14.944423051653523|
|    Manhattan|10.792468572351568|
+-------------+------------------+



In [20]:
# highest/lowest faire amounts for a trip, what burough is associated with the each
print("Highest fare trip:")
df_joined.orderBy(F.desc('fare_amount')).limit(1) \
         .select('Borough', 'fare_amount').show()

print("Lowest fare trip:")
df_joined.orderBy('fare_amount').limit(1) \
         .select('Borough', 'fare_amount').show()

Highest fare trip:
+---------+-----------+
|  Borough|fare_amount|
+---------+-----------+
|Manhattan|  623259.86|
+---------+-----------+

Lowest fare trip:
+-------+-----------+
|Borough|fare_amount|
+-------+-----------+
| Queens|     -362.0|
+-------+-----------+



In [21]:
# load the dataset from the most recently available january, is there a change to any of the average metrics.
download_url_2026 = 'https://d37ci6vzurychx.cloudfront.net/trip-data/yellow_tripdata_2026-01.parquet'

response = requests.get(download_url_2026)
jan_2026_trip_data = "yellow_tripdata_2026-01.parquet"
if response.status_code == 200:
    with open(jan_2026_trip_data, "wb") as f:
        f.write(response.content)

df_trips_2026 = spark.read.parquet(jan_2026_trip_data)

old_avg = df_trips.agg(
    F.avg('trip_distance').alias('avg_distance'),
    F.avg('fare_amount').alias('avg_fare')
).collect()[0]

new_avg = df_trips_2026.agg(
    F.avg('trip_distance').alias('avg_distance'),
    F.avg('fare_amount').alias('avg_fare')
).collect()[0]

print(f"2019 - avg distance: {old_avg['avg_distance']:.2f}, avg fare: {old_avg['avg_fare']:.2f}")
print(f"2026 - avg distance: {new_avg['avg_distance']:.2f}, avg fare: {new_avg['avg_fare']:.2f}")

2019 - avg distance: 2.83, avg fare: 12.53
2026 - avg distance: 6.46, avg fare: 20.80


### Part 3

- choose 3 questions from above and re-answer them using the language you did not use for the main notebook . (i.e - if you completed the exercise in python, redo 3 questions in pure sql) . at least one of the questions to be redone must involve a join

In [22]:
df_trips.createOrReplaceTempView("trips")

In [23]:
# What is the Average passanger count (SQL version)
spark.sql("""
    SELECT AVG(passenger_count) AS avg_passenger_count
    FROM trips
""").show()

+-------------------+
|avg_passenger_count|
+-------------------+
| 1.5670317144945614|
+-------------------+



In [24]:
# Which trip has the highest passanger count (SQL version)
spark.sql("""
    SELECT *
    FROM trips
    ORDER BY passenger_count DESC
    LIMIT 1
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount|extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|         ID|trip_duration_seconds|pickup_date|pickup_hour|time_of_day|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+-----+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------+--

In [25]:
# What was the highest "extra" charge and which trip (SQL version)
spark.sql("""
    SELECT *
    FROM trips
    ORDER BY extra DESC
    LIMIT 1
""").show()

+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------+-----------+-----------+
|VendorID|tpep_pickup_datetime|tpep_dropoff_datetime|passenger_count|trip_distance|RatecodeID|store_and_fwd_flag|PULocationID|DOLocationID|payment_type|fare_amount| extra|mta_tax|tip_amount|tolls_amount|improvement_surcharge|total_amount|congestion_surcharge|airport_fee|         ID|trip_duration_seconds|pickup_date|pickup_hour|time_of_day|
+--------+--------------------+---------------------+---------------+-------------+----------+------------------+------------+------------+------------+-----------+------+-------+----------+------------+---------------------+------------+--------------------+-----------+-----------+---------------------+-----------

# Where to go from here

- Continue building the dataset by loading in more data, start by completing the data for 2019 and calculating the busiest season (fall, winter, spring, summer)
- As of spark v4 dataframes have native visualization support. Choose at least 3 questions from above and provide visualizations.
- Explore a dataset/datasets of your choosing